# Nifty 50 Swing Bot — Backtest (Colab GPU)

Run this whenever you want a full, fresh walk-forward backtest with Optuna
tuning on GPU — without deploying anything. It:

1. Mounts Drive, clones/pulls the repo, restores `models/`/`registry/` state
2. Fetches fresh price data through **today**
3. Runs Optuna hyperparameter search + the purged walk-forward backtest
4. Prints OOF IC / Sharpe / cost sensitivity and writes a detailed
   `reports/backtest_<version>.{json,md}` report (findings, regime overlay
   activity, retrain recommendation)
5. Plots the equity curve
6. Syncs `models/`, `registry/`, `reports/`, `outputs/` back to Drive

This notebook does **not** touch the production model pointer — it never
calls the champion/challenger promotion gate. Use `colab_weekly.ipynb` for
the retrain-and-maybe-deploy job.

In [ ]:
# ── Cell 1: GPU check + environment setup ───────────────────────────────────────────
import os
import subprocess
import logging
from datetime import date

# No logging.basicConfig() exists anywhere in src/ — without a handler, only
# WARNING+ surfaces, so the walk-forward progress/ETA logging is silently
# dropped. Configure it here so backtest progress is actually visible.
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s', force=True)

result = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
print(result.stdout or 'No GPU found — will use CPU (slower training)')

REPO_DIR   = '/content/swing_bot'
DRIVE_BASE = '/content/drive/MyDrive/swing_outputs'
RUN_DATE   = date.today().strftime('%Y-%m-%d')

HORIZON = 5       # trading days
TRIALS  = 50      # Optuna trials (reduce to 10-15 for a quick smoke test)

print(f'Config: horizon={HORIZON}d | trials={TRIALS}')
print(f'Drive base: {DRIVE_BASE}')

In [ ]:
# ── Cell 2: Mount Drive + clone/update repo + restore prior state ───────────
from google.colab import drive
drive.mount('/content/drive')

import re, shutil
from pathlib import Path

GITHUB_REPO = 'https://github.com/aditya33agrawal/XGBoost-Swing-Trading-Prediction-System.git'   # ← UPDATE THIS

if not Path(REPO_DIR).exists():
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
print(f'Repo ready at {REPO_DIR}')

# Restore the most recent best_params / trained model so Optuna has a warm
# starting point context and fast-signals-style runs elsewhere keep working;
# this backtest does its own fresh tuning regardless.
base = Path(DRIVE_BASE)
_date_re = re.compile(r'^\d{4}-\d{2}-\d{2}$')
date_dirs = sorted(
    [p for p in base.glob('*') if p.is_dir() and _date_re.match(p.name)],
    key=lambda p: p.name,
) if base.exists() else []
restore_from = date_dirs[-1] if date_dirs else (base if base.exists() else None)
print(f'Restoring previous state from: {restore_from}' if restore_from else 'No previous runs on Drive — starting fresh')

for folder in ('models', 'registry'):
    dst = Path(REPO_DIR) / folder
    src = (restore_from / folder) if restore_from is not None else None
    if src is not None and src.exists():
        shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
        print(f'  restored {folder}/')
    else:
        dst.mkdir(parents=True, exist_ok=True)

%cd {REPO_DIR}
import sys; sys.path.insert(0, REPO_DIR)

In [ ]:
# ── Cell 3: Install dependencies ──────────────────────────────────────
!pip install -q optuna yfinance plotly python-dotenv
# pandas-ta skipped on Colab (numpy 2.x incompatibility) — the pipeline falls
# back to its own TA implementations when it's missing.

import xgboost as xgb
print(f'XGBoost {xgb.__version__}')
try:
    import numpy as np
    m = xgb.XGBClassifier(device='cuda', n_estimators=1)
    m.fit(np.random.randn(10, 2), np.random.randint(0, 2, 10))
    print('GPU (CUDA) training: AVAILABLE')
except Exception as e:
    print(f'GPU not available ({e}) — using CPU')

In [ ]:
# ── Cell 4: Run the full backtest (fresh data through today, Optuna tuning) ──
from src.config import Config
from src.pipeline.runner import run
from src.models.improvement import get_model_version

run_id = get_model_version()
print(f'Backtest run_id: {run_id}')

cfg = Config(
    horizon          = HORIZON,
    xgb_n_trials     = TRIALS,
    skip_backtest    = False,
    rebalance_every  = HORIZON,
    embargo          = HORIZON,
    model_version    = run_id,
    save_outputs     = True,
    save_to_supabase = False,    # backtest-only run; no DB writes
    paper_trade      = False,    # pure research backtest, no paper-trade bookkeeping
    resolve_outcomes_on_start = False,
    # cfg.end defaults to today() — do not hardcode an end date here.
)

stats, signals = run(cfg)

In [ ]:
# ── Cell 5: Plot the equity curve ─────────────────────────────────────
import matplotlib.pyplot as plt

equity = stats.get('equity_curve')
if equity is not None and len(equity):
    plt.figure(figsize=(11, 4))
    equity.plot()
    plt.title(f"OOF equity curve — {run_id}  (Sharpe={stats.get('Sharpe')}, CAGR={stats.get('CAGR')}, maxDD={stats.get('max_drawdown')})")
    plt.ylabel('Equity (× initial)')
    plt.grid(alpha=0.3)
    plt.show()
else:
    print('No equity curve in stats — check stats.get("error")', stats.get('error'))

In [ ]:
# ── Cell 6: Show the detailed backtest report (findings, cost sensitivity, drift) ──
import json
from pathlib import Path

report_path = Path(cfg.reports_dir) / f'backtest_{run_id}.md'
if report_path.exists():
    print(report_path.read_text())
else:
    print(f'No report found at {report_path} — check the run output above for errors.')

In [ ]:
# ── Cell 7: Sync everything this run produced back to Drive ─────────────
drive_run = Path(DRIVE_BASE) / RUN_DATE
drive_run.mkdir(parents=True, exist_ok=True)

for folder in ('models', 'registry', 'reports', 'outputs'):
    src = Path(REPO_DIR) / folder
    if src.exists():
        shutil.copytree(str(src), str(drive_run / folder), dirs_exist_ok=True)
        print(f'  synced {folder}/ → {drive_run / folder}')

print('\nDone. This backtest did not touch the production model pointer.')
print('To retrain AND deploy if it wins, run colab_weekly.ipynb instead.')